# Notebook 03: Analisis estadistico de los datos

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42

PROYECTO = "food_delivery"

EN_DRIVE = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError(
            "Drive no quedo montado. Volve a ejecutar esta celda y autoriza el acceso "
            "en la ventana emergente."
        )
    RUTA = f"/content/drive/MyDrive/{PROYECTO}"
    EN_DRIVE = True
except ImportError:
    RUTA = os.path.abspath(f"./{PROYECTO}")

CARPETA_SPLITS = os.path.join(RUTA, "splits")
os.makedirs(CARPETA_SPLITS, exist_ok=True)


def guardar(nombre):
    destino = os.path.join(CARPETA_SPLITS, nombre + ".png")
    plt.savefig(destino, dpi=150, bbox_inches="tight")
    print("Figura guardada:", destino)


print("Persistencia en Drive:", EN_DRIVE)
print("Ruta de trabajo:", RUTA)

In [ ]:
X_train = pd.read_csv(os.path.join(RUTA, "X_train.csv"))
X_test = pd.read_csv(os.path.join(RUTA, "X_test.csv"))
y_train = pd.read_csv(os.path.join(RUTA, "y_train.csv")).iloc[:, 0]
y_test = pd.read_csv(os.path.join(RUTA, "y_test.csv")).iloc[:, 0]

NUMERICAS = ["distancia_km", "edad", "calificacion", "pedidos_simultaneos", "estado_vehiculo"]
CATEGORICAS = ["trafico", "clima", "vehiculo", "tipo_pedido", "ciudad", "festivo"]

print("Entrenamiento:", X_train.shape, " Prueba:", X_test.shape)

In [ ]:
train = X_train.copy()
train["minutos"] = y_train.values
print("Observaciones para el analisis:", len(train))

## 4.1 Estadistica descriptiva

Se reportan media, mediana, desviacion estandar, minimo, maximo y cuartiles para la variable
objetivo y para las cinco predictoras numericas.

In [ ]:
descriptiva = train[NUMERICAS + ["minutos"]].describe().T
descriptiva = descriptiva.rename(columns={
    "count": "n", "mean": "media", "std": "desv", "min": "minimo",
    "25%": "Q1", "50%": "mediana", "75%": "Q3", "max": "maximo"})
descriptiva[["n", "media", "mediana", "desv", "minimo", "Q1", "Q3", "maximo"]].round(2)

**Lectura de la variable objetivo.** El tiempo de entrega promedia unos 26 minutos con una
mediana practicamente igual, lo que indica una distribucion sin cola larga: no hay un grupo
de entregas extremadamente lentas arrastrando el promedio. El rango va de 10 a 54 minutos, de
modo que toda la variacion que hay que explicar cabe en 44 minutos. Ese dato importa para
leer las metricas de error mas adelante: un error de 6 minutos sobre un rango de 44 es
distinto de un error de 6 minutos sobre un rango de 200.

La desviacion estandar cercana a 9 minutos es la referencia contra la cual se mide cualquier
modelo. Un modelo que no aprendiera nada y siempre predijera la media cometeria un error
cuadratico medio igual a esa varianza. Bajar de ahi es lo minimo exigible.

**Lectura de las predictoras.** La distancia se concentra entre 5 y 14 kilometros, con
mediana cercana a 9. Es un rango estrecho, y esa estrechez es justamente lo que la hipotesis
anticipa como causa de que la distancia no domine: si todas las entregas recorren distancias
parecidas, la distancia no puede explicar por que unas tardan el doble que otras.

La calificacion de los repartidores esta comprimida entre 2,5 y 5 con mediana en 4,7. Casi
todos estan bien calificados, de modo que la variable discrimina poco en la mayor parte de su
rango y solo separa a una minoria mal evaluada.

In [ ]:
categorias = []
for c in CATEGORICAS:
    g = train.groupby(c, observed=True)["minutos"].agg(["count", "mean", "std"])
    g.index.name = None
    g = g.assign(variable=c).reset_index().rename(columns={"index": "categoria"})
    categorias.append(g)

tabla_cat = pd.concat(categorias)[["variable", "categoria", "count", "mean", "std"]]
tabla_cat.columns = ["variable", "categoria", "n", "minutos_medios", "desv"]
tabla_cat = tabla_cat.reset_index(drop=True)

In [ ]:
print(tabla_cat.round(2).to_string(index=False))

**Lectura de las categoricas.** Tres cosas saltan de esta tabla.

El **trafico** ordena los tiempos exactamente como cabria esperar y con separaciones amplias:
entre transito bajo y atasco hay cerca de diez minutos de diferencia, casi la cuarta parte del
rango completo de la variable objetivo. Ademas cada uno de sus cuatro niveles reune miles de
observaciones, de modo que esas medias son solidas.

**Los dos saltos mas grandes vienen de categorias muy poco frecuentes.** La ciudad semiurbana
casi duplica el tiempo medio, y el dia festivo lo aumenta en unos veinte minutos, pero entre
las dos suman menos del tres por ciento de los pedidos. Son efectos intensos y raros: importan
mucho para predecir esos casos puntuales y poco para el desempeno promedio, porque casi nunca
se activan. Conviene no confundir el tamano del salto con la importancia de la variable, y por
eso la seccion siguiente mide cuanta varianza explica cada una en lugar de mirar solo la
diferencia entre sus extremos.

El **tipo de pedido** no separa nada. Bebidas, buffet, comida y snack difieren en menos de
medio minuto entre si. Es un hallazgo negativo y vale reportarlo: descarta la idea intuitiva
de que un buffet tarda mas en prepararse que una bebida. La explicacion mas probable es que el
tiempo de preparacion queda absorbido dentro del intervalo entre el pedido y la recogida, que
este experimento no modela.

## 4.2 Analisis de correlacion y visualizacion

### Matriz de correlacion

Se calcula sobre las variables numericas. La correlacion de Pearson mide asociacion lineal
entre variables continuas, de modo que las categoricas quedan fuera por construccion. Esa
limitacion no es un detalle tecnico: es central para la lectura de esta seccion, y se retoma
mas abajo.

In [ ]:
correlaciones = train[NUMERICAS + ["minutos"]].corr()

plt.figure(figsize=(7.5, 6))
sns.heatmap(correlaciones, annot=True, fmt=".3f", cmap="Greys", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Matriz de correlacion de las variables numericas")
plt.tight_layout()
guardar("03_matriz_correlacion")
plt.show()

In [ ]:
con_objetivo = correlaciones["minutos"].drop("minutos")
con_objetivo = con_objetivo.reindex(con_objetivo.abs().sort_values(ascending=False).index)
print("Correlacion de cada predictora numerica con el tiempo de entrega:")
print(con_objetivo.round(3).to_string())

**Lectura de la matriz.** Ninguna correlacion con el tiempo de entrega supera 0,4 en valor
absoluto. La distancia, que la intuicion senalaria como el predictor principal, queda por
debajo de los pedidos simultaneos y de la calificacion del repartidor. Los signos son los
esperados: mas distancia y mas pedidos a la vez alargan la entrega, mientras que un
repartidor mejor calificado y un vehiculo en mejor estado la acortan.

**Lo importante es lo que la matriz no puede mostrar.** Todas las correlaciones son debiles,
pero eso no significa que el tiempo de entrega sea impredecible. Significa que los predictores
fuertes son categoricos y por lo tanto no aparecen aca: el trafico separa diez minutos y el
dia festivo veinte, como se vio en la tabla anterior, y ninguno de los dos puede entrar en un
coeficiente de Pearson. Interpretar esta matriz por si sola llevaria a concluir que el
problema no tiene solucion, y seria una conclusion equivocada.

Entre predictoras no hay correlaciones altas, de modo que no hay multicolinealidad de la que
preocuparse. Los coeficientes del modelo lineal van a ser estables e interpretables.

### Cuanta varianza explica cada factor

Para poder comparar en la misma escala variables numericas y categoricas se calcula, para
cada una por separado, la proporcion de la varianza del tiempo de entrega que explica. Para
las numericas es el cuadrado de la correlacion; para las categoricas es la razon entre la
varianza de las medias de grupo y la varianza total. Ambas cantidades responden a la misma
pregunta: si conociera solo esta variable, cuanto del tiempo podria explicar.

**Este calculo es el que decide la segunda parte de la hipotesis.**

In [ ]:
varianza_total = ((train["minutos"] - train["minutos"].mean()) ** 2).sum()

filas = []
for c in NUMERICAS:
    filas.append({"variable": c, "tipo": "numerica",
                  "varianza_explicada": train[c].corr(train["minutos"]) ** 2})
for c in CATEGORICAS:
    g = train.groupby(c, observed=True)["minutos"]
    entre = (((g.mean() - train["minutos"].mean()) ** 2) * g.count()).sum()
    filas.append({"variable": c, "tipo": "categorica",
                  "varianza_explicada": entre / varianza_total})

aporte = pd.DataFrame(filas).sort_values("varianza_explicada", ascending=False)
aporte["varianza_explicada"] = aporte["varianza_explicada"].round(4)
print(aporte.to_string(index=False))

In [ ]:
plt.figure(figsize=(8, 4.5))
colores = ["#404040" if t == "categorica" else "#a0a0a0" for t in aporte["tipo"]]
plt.barh(aporte["variable"], aporte["varianza_explicada"], color=colores)
plt.gca().invert_yaxis()
plt.xlabel("Proporcion de la varianza del tiempo de entrega explicada")
plt.title("Aporte individual de cada variable")
for i, v in enumerate(aporte["varianza_explicada"]):
    plt.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=9)
plt.tight_layout()
guardar("03_varianza_por_variable")
plt.show()

**Lectura.** El trafico encabeza el ranking y explica cerca del doble que la distancia. Esa
comparacion es exactamente la segunda parte de la hipotesis, y se contrasta formalmente en el
notebook 05.

**Pero el trafico no es lo unico que le gana a la distancia.** Los pedidos simultaneos y la
calificacion del repartidor tambien la superan. Es decir que la carga de trabajo del
repartidor y quien lleva el pedido pesan mas sobre el tiempo final que cuan lejos queda el
destino. La distancia recien aparece en el cuarto lugar de once variables.

Ese resultado ordena las prioridades de cualquier mejora: para estimar mejor el tiempo
conviene medir bien las condiciones de circulacion y la carga del repartidor antes que refinar
el calculo de la ruta.

En el otro extremo, el tipo de pedido explica una fraccion despreciable de la varianza,
consistente con lo que ya mostraba la tabla de medias. Es una variable que se puede quitar del
modelo sin costo, y se conserva unicamente para dejar documentado el hallazgo negativo.

### Diagramas de dispersion

El grafico siguiente enfrenta la distancia contra el tiempo de entrega, separando los puntos
por densidad de trafico. Es la visualizacion que decide la lectura del problema: si la
distancia dominara, los puntos formarian una nube ascendente compacta; si dominara el
trafico, se separarian en bandas horizontales por color.

In [ ]:
muestra = train.sample(n=min(4000, len(train)), random_state=RANDOM_STATE)
orden = ["Low", "Medium", "High", "Jam"]
etiquetas = {"Low": "Trafico bajo", "Medium": "Trafico medio",
             "High": "Trafico alto", "Jam": "Atasco"}

fig, ejes = plt.subplots(1, 2, figsize=(13, 5))

for nivel, color in zip(orden, ["#c8c8c8", "#909090", "#585858", "#101010"]):
    sub = muestra[muestra["trafico"] == nivel]
    ejes[0].scatter(sub["distancia_km"], sub["minutos"], s=7, alpha=0.45,
                    color=color, label=etiquetas[nivel])
ejes[0].set_xlabel("Distancia (km)")
ejes[0].set_ylabel("Tiempo de entrega (min)")
ejes[0].set_title("Distancia contra tiempo, separado por trafico")
ejes[0].legend(markerscale=2.5, fontsize=9)

sns.regplot(data=muestra, x="distancia_km", y="minutos", ax=ejes[1],
            scatter_kws={"s": 7, "alpha": 0.25, "color": "#909090"},
            line_kws={"color": "#101010", "linewidth": 2})
ejes[1].set_xlabel("Distancia (km)")
ejes[1].set_ylabel("Tiempo de entrega (min)")
ejes[1].set_title("Ajuste lineal simple sobre la distancia")

plt.tight_layout()
guardar("03_dispersion_distancia_trafico")
plt.show()

**Lectura de la dispersion.** El panel derecho muestra la recta de regresion sobre la
distancia sola: tiene pendiente positiva, lo que confirma la direccion de la hipotesis, pero
la nube de puntos es tan ancha que la recta explica una fraccion minima de lo que se observa.
Para cualquier distancia fija hay entregas de quince minutos y entregas de cuarenta.

El panel izquierdo explica esa dispersion. Los puntos no estan mezclados al azar: se apilan
en bandas segun el trafico, con los atascos arriba y el transito bajo abajo, y las bandas se
mantienen separadas a lo largo de todo el eje de distancias. Buena parte del ancho de la nube
del panel derecho no es ruido, es trafico sin modelar.

### Distribuciones

Tres vistas complementarias: la forma de la variable objetivo, como se desplaza segun el
trafico y como se desplaza segun el clima.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.4))

ejes[0].hist(train["minutos"], bins=40, color="#707070", edgecolor="white")
ejes[0].axvline(train["minutos"].mean(), color="#101010", linestyle="--", linewidth=1.6,
                label=f"media {train['minutos'].mean():.1f}")
ejes[0].set_xlabel("Tiempo de entrega (min)")
ejes[0].set_ylabel("Pedidos")
ejes[0].set_title("Distribucion del tiempo de entrega")
ejes[0].legend(fontsize=9)

sns.boxplot(data=train, x="trafico", y="minutos", order=orden, ax=ejes[1],
            color="#b0b0b0", fliersize=1.5)
ejes[1].set_xticks(range(len(orden)))
ejes[1].set_xticklabels([etiquetas[o] for o in orden], rotation=20, ha="right")
ejes[1].set_xlabel("")
ejes[1].set_ylabel("Tiempo de entrega (min)")
ejes[1].set_title("Tiempo segun densidad de trafico")

orden_clima = train.groupby("clima", observed=True)["minutos"].mean().sort_values().index
sns.boxplot(data=train, x="clima", y="minutos", order=orden_clima, ax=ejes[2],
            color="#b0b0b0", fliersize=1.5)
ejes[2].set_xticks(range(len(orden_clima)))
ejes[2].set_xticklabels(orden_clima, rotation=20, ha="right")
ejes[2].set_xlabel("")
ejes[2].set_ylabel("")
ejes[2].set_title("Tiempo segun condicion climatica")

plt.tight_layout()
guardar("03_distribuciones")
plt.show()

**Lectura de las distribuciones.** El histograma muestra una variable aproximadamente
simetrica y sin cola larga, lo que respalda usar el error cuadratico medio como criterio: esa
metrica penaliza fuerte los errores grandes, y seria una mala eleccion si hubiera un grupo de
entregas extremadamente lentas dominando la penalizacion.

Los dos diagramas de caja muestran cajas que se desplazan hacia arriba de manera ordenada al
empeorar las condiciones, y con solapamiento parcial entre niveles contiguos. Ese solapamiento
es esperable y anticipa el techo del modelo: conocer el nivel de trafico desplaza la
prediccion en la direccion correcta pero no determina el resultado, porque dentro de cada
nivel sigue habiendo variacion que estas variables no capturan.

### La tabla que resume el hallazgo

Para separar limpiamente el efecto de la distancia del efecto del trafico se cruzan ambas:
tramos de distancia en las filas, niveles de trafico en las columnas, tiempo medio en las
celdas. Leer una fila muestra cuanto cambia el tiempo por el trafico manteniendo la distancia
aproximadamente fija; leer una columna muestra cuanto cambia por la distancia manteniendo el
trafico fijo.

In [ ]:
train["tramo"] = pd.cut(train["distancia_km"], [0, 5, 10, 15, 30],
                        labels=["0-5 km", "5-10 km", "10-15 km", "15-30 km"])

cruce = train.pivot_table(index="tramo", columns="trafico", values="minutos",
                          aggfunc="mean", observed=True)
cruce = cruce.reindex(columns=[c for c in orden if c in cruce.columns])
cruce.columns = [etiquetas[c] for c in cruce.columns]

print("Tiempo medio de entrega en minutos:")
print(cruce.round(1).to_string())

if "Atasco" in cruce.columns and "Trafico bajo" in cruce.columns:
    print("\nDiferencia entre atasco y trafico bajo, dentro de cada tramo de distancia:")
    print((cruce["Atasco"] - cruce["Trafico bajo"]).round(1).to_string())

print("\nDiferencia entre el tramo mas largo y el mas corto, con trafico bajo:")
col = "Trafico bajo"
print(f"  {(cruce[col].iloc[-1] - cruce[col].iloc[0]).round(1)} minutos")

**El hallazgo central del analisis.** Dentro de un mismo tramo de distancia, pasar de transito
bajo a atasco agrega entre siete y diez minutos. Recorrer todo el rango de distancias
manteniendo el trafico bajo agrega bastante menos que eso, y entre los dos primeros tramos
practicamente no agrega nada: cinco kilometros adicionales dejan el tiempo medio donde estaba.

Algunas celdas quedan vacias porque esa combinacion no aparece en los datos: no hay pedidos
largos registrados con trafico alto. Es una limitacion del conjunto, no un error de calculo, y
conviene tenerla presente al leer la tabla.

Traducido a lo que significa para una plataforma de reparto: el tiempo que se le promete al
cliente no deberia calcularse a partir de la distancia. Un pedido a dos kilometros en hora
pico llega despues que uno a ocho kilometros con la avenida libre. Es una afirmacion que el
notebook 05 va a contrastar formalmente contra la hipotesis, con los coeficientes del modelo
en la mano.